# Projeto: Boletim Climático

## Descrição Geral

Este notebook automatiza a geração e o envio de boletins climáticos personalizados por município, com base em previsões meteorológicas e alertas de alagamento. Os boletins são enviados diariamente, a partir das 07h, para usuários cadastrados na [Aqualert Newsletter](https://crescerfunction.azurewebsites.net/api/aqualert/receber_alertas).<br>
O processo é totalmente automatizado e segmentado por destinatário, contemplando extração, processamento, formatação e envio em um fluxo contínuo.


## Objetivos

- Automatizar a produção de boletins por município com base em dados atualizados.
- Garantir escalabilidade e modularidade na arquitetura de geração e envio.
- Minimizar complexidade de manutenção e facilitar evolução futura.
- Documentar claramente a arquitetura e responsabilidades envolvidas.

## Tecnologias Utilizadas

- **Apache Spark (PySpark)** – Processamento distribuído de dados climáticos.
- **Azure Blob Storage** – Armazenamento de emails cadastrados.
- **Databricks Notebooks** – Ambiente de orquestração e execução automatizada.
- **Rocket.Chat API (CWI Rocket)** – Enriquecimento e formatação via workspaces dedicadas.
- **SMTP (Gmail)** – Envio programático de emails personalizados.

---

## Arquitetura e Responsabilidades

- **Engine de comunicação com Rocket.Chat** – Desenvolvida por *Gabriel Rost*. Responsável por encapsular chamadas às workspaces de previsão, alagamento e formatação.
- **Orquestração da geração e envio de boletins** – Desenvolvida por *Evelise*. Responsável por agrupar cidades por email, iterar sobre cadastros, gerar conteúdo e executar os envios com tratamento de exceções.
- **Compatibilidade com cidades acentuadas** – Implementada por *Lucas Berta*. Corrigiu problemas com nomes contendo caracteres especiais.

---

## Histórico de Atualizações

| Data       | Responsável     | Versão | Descrição                                                                 |
|------------|------------------|--------|--------------------------------------------------------------------------|
| 25/05/2025 | Gabriel Rost      | 1.0    | Implementação da engine de comunicação com Rocket.Chat                   |
| 25/05/2025 | Evelise           | 1.1    | Orquestração de geração de boletins segmentados por email                |
| 25/05/2025 | Lucas Berta       | 1.2    | Correção de compatibilidade com cidades acentuadas                       |


In [0]:
%run ../utils/utils_run_rocket

In [0]:
path_emails = landingzone_cadastro_emails
df_cadastro_emails = spark.read.format("csv").option("header", "true").load(path_emails)

In [0]:
workspace_previsao_data_determinada = "aqualert"
workspace_analise_alagamentos = "alertaalagamentos"
workspace_formatar_email = "formatarnoticia"

In [0]:
def gerar_boletim_para_email(email, cidades):
    """
    Gera e envia o conteúdo do boletim climático para um destinatário específico, consolidando as informações de múltiplas cidades em um único email.

    Parâmetros:
    - email (str): Endereço de email do destinatário.
    - cidades (List[str]): Lista de nomes de municípios associados ao email.

    Processo:
    1. Para cada cidade:
    - Busca previsões climáticas e alertas de alagamento atualizados.
    - Converte os dados em mensagens formatadas via Rocket.Chat.
    2. Concatena os conteúdos de todas as cidades em uma estrutura unificada de boletim.
    3. Formata o boletim completo com a engine de formatação.
    4. Envia o email com o boletim personalizado.

    """
    data_atual = date.today()
    conteudo_total = ""

    for cidade in cidades:
        ## Previsão do Clima Para o Dia Atual
        df_previsao = (
            ler_tabelas("grupo01.gold.previsao_clima")
            .filter(
                (col("dt_previsao") == data_atual) & (col("nom_municipio") == cidade)
            )
            .select(
                "dt_carga",
                "dt_previsao",
                "nom_municipio",
                "nom_estado",
                "temp_max",
                "temp_min",
                "umid_max",
                "umid_min",
                "precipitacao",
                "prob_precipitacao",
                "vel_vento",
                "flg_fonte_previsao",
            )
        )
        mensagem_previsao = converter_para_rocket(df_previsao)
        
        # Análises de Possíveis Alagamentos Para os Próximos Cinco Dias
        df_alagamento = (
            ler_tabelas("grupo01.gold.alertas_alagamentos")
            .filter(
                (col("nom_municipio") == cidade)
                & (col("dt_previsao") >= current_date())
                & (col("dt_previsao") <= expr("current_date() + interval 5 days"))
            )
            .select(
                "dt_previsao",
                "nom_municipio",
                "nom_estado",
                "pais",
                "vol_max_chuva",
                "precipitacao_prevista",
                "prob_precipitacao_prevista",
                "flg_fonte_previsao",
            )
        )
        mensagem_alagamento = converter_para_rocket(df_alagamento)

        # Chats com Rocket
        rocket_previsao = chat_rocket(
            workspace_previsao_data_determinada, mensagem_previsao
        )
        rocket_alagamento = chat_rocket(
            workspace_analise_alagamentos, mensagem_alagamento
        )

        #Cria coporto do email
        conteudo = f"""
        \nINICIO DO CABEÇALHO\n" + 
        "{rocket_previsao} - {cidade}" + 
        "\nFIM DO CABEÇALHO\n" + 
        
        "(A Seguir As Próximas Sessões Apresentam Previsões Para Demais Cidades Do Brasil, Informar Período das Previsões)\n" + 
        "\nINICIO DO CONTEUDO\n" + 
        "{rocket_alagamento}" 
        + "\nFIM DO CONTEUDO\n"""

        conteudo_total += conteudo + "\n\n"

    conteudo_formatado = chat_rocket(workspace_formatar_email, conteudo_total)

    #Disparo de email
    enviar_email(
        [email],
        f"Boletim Climático - {', '.join(cidades)} - {data_atual}",
        conteudo_formatado,
    )

In [0]:
def gerar_previsao():
    """
    Orquestra o envio de boletins climáticos individualizados para cada destinatário com base na lista de inscrições.

    Processo:
    - Lê a base de emails cadastrados, contendo mapeamentos entre cidades e destinatários.
    - Agrupa as cidades por email, consolidando múltiplos municípios para o mesmo usuário.
    - Para cada destinatário, chama a função `gerar_boletim_para_email`, que monta e envia o boletim completo.
    - Cada envio é encapsulado em um bloco de tratamento de exceções, garantindo que falhas individuais não interrompam o processo global.
    """

    # Agrupa cidades por email
    emails_cidades = (
        df_cadastro_emails
        .groupBy("email")
        .agg(collect_set("cidade").alias("cidades"))
        .collect() 
    )

        # Itera sobre cada email + lista de cidades
    for linha in emails_cidades:
        email = linha["email"]
        cidades = linha["cidades"]

        print(f"Enviando boletim para {email} - cidades: {cidades}")

        try:
            gerar_boletim_para_email(email, cidades)
        except Exception as e:
            print(f"Erro ao processar {email}: {e}")